# Animals interference — Polygram v0 walking tour

This notebook mirrors `examples/animals_interference.py` with explanatory cells and final plots. It declares a 4-feature, 2-cluster Animals dictionary (`dog_poodle`, `dog_beagle`, `bird_hawk`, `bird_sparrow`) and watches the cross-cluster `(dog_poodle, bird_hawk)` overlap as we sweep `bird_hawk.phi` (1D) — and as a follow-up, on a 2D grid of `(dog_poodle.phi, bird_hawk.phi)`.

**What to look for**

- *Hierarchy preserved.* In-cluster sibling overlaps stay above the cross-cluster target overlap at every sweep point.
- *Single-φ does not cancel.* On this rung-1 MPS geometry, sweeping `bird_hawk.phi` alone does **not** drive `(dog_poodle, bird_hawk)` to zero — the overlap actually rises with φ, peaking near π. Driving destructive interference needs antisymmetric φ on **both** sides; that is the future `Cancellation` primitive's territory, not v0.
- *2D grid reveals the asymmetry.* The heatmap of overlap vs `(φ_dog, φ_bird)` makes the antisymmetric direction visible — overlap drops along the antidiagonal, rises along the diagonal.
- *Verifiable artifact.* `experiment.materialize(...)` writes a `.q.orca.md` machine that parses + verifies clean against `q-orca>=0.7.1`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from examples.animals_interference import build_dictionary, build_experiment

## 1D sweep — `bird_hawk.phi` from 0 to π

`build_dictionary()` returns a 4-feature `Dictionary` with two clusters (`dogs`, `birds`) and an `MPSRung1` encoding (bond-2, phase knobs enabled). `build_experiment(...)` wraps it into a phase sweep targeting the cross-cluster pair.

In [ ]:
dictionary = build_dictionary()
experiment = build_experiment(dictionary, n_points=40, mode="single")

out = Path("output/animals_interference")
experiment.materialize(out)
result = experiment.run()
result.write_summary(out / f"{experiment.name}_summary.md")

print(f"emitted {out / (experiment.name + '.q.orca.md')}")
print(f"sweep:   {dict((k, len(v)) for k, v in result.sweep_axes.items())}")
print(f"overlap range: [{result.overlaps.min():.4f}, {result.overlaps.max():.4f}]")
print(f"sibling tier mean: {float(np.nanmean(result.tier_stats['sibling'])):.4f}")
print(f"cross   tier mean: {float(np.nanmean(result.tier_stats['cross_cluster'])):.4f}")

### Plot — target overlap with tier baselines

`result.plot(...)` saves the figure to disk; the inline render below uses the same data.

In [ ]:
result.plot(out / f"{experiment.name}_overlap.png")

phis = result.sweep_axes["bird_hawk.phi"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(phis, result.overlaps, marker="o", linewidth=1.5, label="(dog_poodle, bird_hawk)")
ax.plot(phis, result.tier_stats["sibling"], linestyle="--", color="tab:green", label="sibling tier")
ax.plot(phis, result.tier_stats["cross_cluster"], linestyle=":", color="tab:gray", label="cross-cluster tier")
ax.set_xlabel(r"$\varphi_{\mathrm{bird\_hawk}}$")
ax.set_ylabel(r"$|\langle A | B \rangle|^2$")
ax.set_xticks([0, np.pi / 2, np.pi])
ax.set_xticklabels(["0", r"$\pi/2$", r"$\pi$"])
ax.legend(loc="best")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2D sweep — `(dog_poodle.phi, bird_hawk.phi)` grid

Rerunning with `mode="two_axis"` produces a 12×12 grid; the saved heatmap lives at `<out>/AnimalsInterference_overlap.png` (overwriting the 1D plot). Use a fresh subdir when comparing the two.

In [ ]:
experiment2d = build_experiment(dictionary, n_points=12, mode="two_axis")
out2d = Path("output/animals_interference_2d")
experiment2d.materialize(out2d)
result2d = experiment2d.run()
result2d.plot(out2d / f"{experiment2d.name}_overlap.png")

xs = result2d.sweep_axes["dog_poodle.phi"]
ys = result2d.sweep_axes["bird_hawk.phi"]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(result2d.overlaps.T, origin="lower", aspect="auto",
               extent=(xs[0], xs[-1], ys[0], ys[-1]), cmap="viridis")
fig.colorbar(im, ax=ax, label=r"$|\langle A | B \rangle|^2$")
ax.set_xlabel(r"$\varphi_{\mathrm{dog\_poodle}}$")
ax.set_ylabel(r"$\varphi_{\mathrm{bird\_hawk}}$")
ax.set_title("AnimalsInterference — 2D phase landscape")
fig.tight_layout()
plt.show()

## Assertions and tier rollup

`hierarchical_ordering_preserved` checks that in-cluster sibling overlaps dominate the target-pair overlap at every sweep point. The tier rollup (in the saved `_summary.md`) gives min/mean/max per tier across the sweep.

In [ ]:
for name, passes in result.assertion_pass.items():
    print(f"{name}: pass-rate {passes.mean():.0%}")
for name, passes in result2d.assertion_pass.items():
    print(f"2D {name}: pass-rate {passes.mean():.0%}")